## 1. Import thư viện 


In [2]:
import numpy as np #thư viện xử lý số học. Dùng cho các phép toán ma trận, đại số tuyến tính, và các hàm toán học khác.
import matplotlib.pyplot as plt #Vẽ đồ thị
import pandas as pd #Xử lý dữ liệu dạng bảng
import seaborn as sns #thư viện vẽ đồ thị đẹp hơn matplotlib
from sklearn.model_selection import train_test_split, GridSearchCV 
# train_test_split: Chia dữ liệu thành tập huấn luyện và tập kiểm tra
# GridSearchCV: Tìm kiếm tham số tốt nhất cho mô hình bằng cách sử dụng cross-validation
from sklearn.ensemble import RandomForestRegressor #Mô hình rừng ngẫu nhiên cho bài toán hồi quy
import xgboost as xgb #Thư viện XGBoost cho các mô hình boosting
from sklearn.metrics import mean_squared_error, r2_score #Đánh giá hiệu suất mô hình
import lightgbm as lgb #Thư viện LightGBM cho các mô hình boosting
from tqdm.notebook import tqdm #Thanh tiến trình trong Jupyter Notebook
from sklearn.model_selection import ParameterGrid #Tạo lưới tham số để tìm kiếm
from xgboost import XGBRegressor #Mô hình XGBoost cho bài toán hồi quy

## 2. Đọc dữ liệu từ file CSV

In [5]:
df = pd.read_csv('E:\\data_co2_modis.csv') # Đọc dữ liệu từ file CSV vào DataFrame
df.info() 
# hiển thị thông tin: số dòng, số cột, kiểu dữ liệu, và số giá trị không null

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5005 entries, 0 to 5004
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   latitude                   5005 non-null   float64
 1   longitude                  5005 non-null   float64
 2   time                       5005 non-null   object 
 3   date                       5005 non-null   object 
 4   xco2                       5005 non-null   float64
 5   xco2_quality_flag          5005 non-null   int64  
 6   rice_proportion_in_buffer  5005 non-null   float64
 7   is_rice_influenced         5005 non-null   bool   
 8   precipitation              5005 non-null   float64
 9   temperature_2m             4841 non-null   float64
 10  skin_temperature           4841 non-null   float64
 11  soil_temperature_L1        4841 non-null   float64
 12  soil_water_L1              4841 non-null   float64
 13  surface_solar_radiation    4848 non-null   float

## 3. Thống kê mô tả dữ liệu

In [8]:
df.describe()
# tính các thống kê mô tả cơ bản cho các cột số trong DataFrame, bao gồm:
# count: Số lượng giá trị không bị thiếu
# mean: Giá trị trung bình
# std: Độ lệch chuẩn
# min: Giá trị nhỏ nhất
# 25%: Phần trăm thứ 25 (Q1)
# 50%: Phần trăm thứ 50 (Q2, trung vị)
# 75%: Phần trăm thứ 75 (Q3)
# max: Giá trị lớn nhất

,latitude,longitude,xco2,xco2_quality_flag,rice_proportion_in_buffer,precipitation,temperature_2m,skin_temperature,soil_temperature_L1,soil_water_L1,...,NDWI_McFeeters,ET,LE,PET,PLE,LST_Day_Terra_C,LST_Night_Terra_C,LST_Day_Aqua_C,LST_Night_Aqua_C,sm_surface_daily
count,5005.000000,5005.000000,5005.000000,5005.0,5005.0,5005.000000,4841.000000,4841.000000,4841.000000,4841.000000,...,4809.000000,4570.000000,4.570000e+03,4570.000000,4.570000e+03,4030.000000,2443.000000,4800.000000,3002.000000,4935.000000
mean,15.742698,106.800186,413.386584,0.0,1.0,1.906050,24.627223,25.153171,25.732054,0.314345,...,-0.545412,21.726718,6.672411e+06,44.257484,1.362938e+07,32.052615,19.397835,35.963000,19.457302,0.250341
std,4.232644,1.669748,1.847709,0.0,0.0,4.961385,4.014017,4.621535,4.407522,0.093883,...,0.168534,8.495451,2.550928e+06,9.220628,2.718323e+06,5.170066,3.275939,5.781344,3.252429,0.081645
min,9.294680,103.129906,407.509600,0.0,1.0,0.000000,10.621542,10.642893,11.828901,0.056861,...,-0.967848,3.600000,1.100000e+06,21.700001,6.690000e+06,16.370001,7.410000,20.430000,9.850000,0.041705
25%,11.762710,105.278076,412.214260,0.0,1.0,0.000000,22.058731,22.020563,22.416452,0.253494,...,-0.659773,15.700000,4.840000e+06,37.299999,1.155000e+07,28.535000,17.450001,31.969999,17.695000,0.189519
50%,14.410728,107.299070,413.204280,0.0,1.0,0.000000,24.202879,25.163164,25.901894,0.284159,...,-0.586707,21.100000,6.450000e+06,43.000000,1.333000e+07,31.830000,19.650000,35.970001,19.490000,0.250667
75%,20.338816,108.195820,414.504940,0.0,1.0,0.000000,27.448875,28.809433,29.414827,0.404569,...,-0.483537,26.600000,8.120000e+06,50.900002,1.552000e+07,35.085000,21.590000,39.775000,21.590000,0.314131
max,22.980146,109.286980,432.519800,0.0,1.0,30.581598,32.681236,33.939426,33.451366,0.494274,...,0.554167,65.699997,1.999000e+07,69.699997,2.113000e+07,47.650002,25.629999,55.630001,28.450001,0.506450


## 4. Loại bỏ cột và giá trị thiếu

In [ ]:
df_drop = df.drop(columns=['LST_Night_Terra_C', 'LST_Night_Aqua_C'])
df_dropna = df_drop.dropna()
df_dropna.describe()

## 5. Xác định cột đặc trưng (feature) và cột mục tiêu(target)

In [ ]:
feature_columns = df_dropna.columns[df_dropna.columns.get_loc('precipitation'):].tolist()
target = ['xco2']

## 6. Tính tương quan giữa biến đầu vào và đầu ra

In [ ]:
df_selected = df_dropna[feature_columns + target]
correlation_with_target = df_selected.corr()['xco2'].drop('xco2')
print(correlation_with_target)

## 7. Chia dữ liệu train / test


In [ ]:
from sklearn.model_selection import train_test_split

X = df_selected[feature_columns]
y = df_selected['xco2']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Kích thước tập huấn luyện: {X_train.shape}, {y_train.shape}")
print(f"Kích thước tập kiểm tra: {X_test.shape}, {y_test.shape}")

## 8. Tạo grid tham số cho Random Forest

In [ ]:
param_grid_rf = {
    'n_estimators': [250, 300, 350, 400],      
    'max_features': ['sqrt', 'log2', 0.7], 
    'max_depth': [None, 10, 20, 30],       
    'min_samples_split': [2, 5, 10],      
    'min_samples_leaf': [1, 2, 4],         
    'bootstrap': [True]                    
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1) 

grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf,
                              cv=5, 
                              # scoring='neg_mean_squared_error', 
                              scoring='r2',
                              verbose=2, 
                              n_jobs=-1) 

grid_search_rf.fit(X_train, y_train)

print("\nCác siêu tham số tốt nhất cho Random Forest:")
print(grid_search_rf.best_params_)

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"\nĐánh giá Random Forest trên tập kiểm tra:")
print(f"  Mean Squared Error (MSE): {mse_rf:.4f}")
print(f"  R-squared (R2): {r2_rf:.4f}")

## 9. Tạo grid tham số cho XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators': [250, 300, 350, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],          
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],   
    'gamma': [0, 0.1, 0.2],                     
    'reg_alpha': [0, 0.001, 0.01],            # L1 regularization 
    # 'reg_lambda': [1, 0.1, 0.01],           # L2 regularization 
    'tree_method': ['hist'],
    'device': ['cuda']
}

grid = list(ParameterGrid(param_grid_xgb))
best_score = -float('inf')
best_model = None
best_params = None

for params in tqdm(grid, desc="Tuning models"):
    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        **params
    )
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test) 
    if score > best_score:
        best_score = score
        best_model = model
        best_params = params

print("\nCác siêu tham số tốt nhất cho XGBoost:")
print(best_params)

y_pred = best_model.predict(X_test)
mse_xg = mean_squared_error(y_test, y_pred)
r2_xg = r2_score(y_test, y_pred)

print("\nĐánh giá mô hình XGBoost trên tập kiểm tra:")
print(f"  Mean Squared Error (MSE): {mse_xg:.4f}")
print(f"  R-squared (R2): {r2_xg:.4f}")